# ETL Music Pipeline — Demo Notebook

Demonstrates the full ETL pipeline end to end.

In [1]:
from src.extract import extract_csv, extract_musicbrainz
from src.transform import transform_tracks, transform_artists
from src.load import load_tracks, load_artists
from src.config import load_config
import pandas as pd

In [2]:
config = load_config()
sources = config['sources']
print('Sources loaded:', [s['name'] for s in sources])

2026-06-10 10:47:14,770 INFO Loaded config from config/sources.yml


Sources loaded: ['spotify_tracks', 'musicbrainz_artists']


In [3]:
csv_source = next(s for s in sources if s['type'] == 'csv')
df_tracks = extract_csv(csv_source['path'])
print(f'Extracted {len(df_tracks)} rows')
df_tracks.head()

2026-06-10 10:47:15,000 INFO Extracted 114000 rows from data/spotify-tracks-dataset.csv


Extracted 114000 rows


,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [4]:
df_tracks, track_rejects = transform_tracks(df_tracks)
print(f'Clean rows: {len(df_tracks)}')
print(f'Rejected rows: {len(track_rejects)}')
df_tracks.head()

2026-06-10 10:47:15,015 INFO Before cleaning: 114000 rows
2026-06-10 10:47:15,631 INFO After cleaning: 89740 rows | Rejected: 24260 rows


Clean rows: 89740
Rejected rows: 24260


,Unnamed: 0.1,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [5]:
api_source = next(s for s in sources if s['type'] == 'api')
df_artists = extract_musicbrainz(api_source['query'])
print(f'Extracted {len(df_artists)} artists')
df_artists[['name', 'country', 'disambiguation']].head(10)

2026-06-10 10:47:16,297 INFO Extracted 25 artists from MusicBrainz for query: Drake


Extracted 25 artists


,name,country,disambiguation
0,Drake,CA,Canadian rapper
1,Nick Drake,GB,English singer‐songwriter
2,Adam Drake,GB,UK guitarist
3,Bob Drake,NaN,experimental musician
4,Hamid Drake,US,US jazz drummer and percussionist
5,Julius Drake,GB,pianist
6,Drake,NaN,Chilean heavy metal band
7,Drake,NaN,Jason Drake
8,RHMan,NaN,GilvaSunner contributer
9,Drake,NaN,US rapper


In [6]:
df_artists, artist_rejects = transform_artists(df_artists)
print(f'Clean artists: {len(df_artists)}')

2026-06-10 10:47:16,305 INFO Before cleaning artists: 25 rows
2026-06-10 10:47:16,308 INFO After cleaning artists: 25 rows | Rejected: 0 rows


Clean artists: 25


In [7]:
load_tracks(df_tracks, track_rejects)
load_artists(df_artists, artist_rejects)
print('Pipeline complete!')

2026-06-10 10:47:25,739 INFO Loaded 89740 tracks into stg_tracks
2026-06-10 10:47:25,739 INFO Loaded 24260 rejected rows into stg_rejects
2026-06-10 10:47:25,748 INFO Loaded 25 artists into stg_artists
2026-06-10 10:47:25,749 INFO Loaded 0 rejected artists into stg_rejects


Pipeline complete!


In [8]:
print(f'Total tracks: {len(df_tracks)}')
print(f'Total artists: {len(df_artists)}')
print(f'Total rejects: {len(track_rejects)}')
print(df_tracks['track_genre'].value_counts().head(10))

Total tracks: 89740
Total artists: 25
Total rejects: 24260
track_genre
acoustic         1000
afrobeat          999
alt-rock          999
ambient           999
cantopop          999
tango             999
bluegrass         998
chicago-house     998
disney            998
forro             998
Name: count, dtype: int64
